In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('./../tmpsnx71rnm.csv')
data.columns

In [ ]:
data.drop(columns=['Unnamed: 0', 'Zero-shot','Active Parameters (B)','Classification', 'Clustering', 'Instruction Reranking',
       'Multilabel Classification', 'Pair Classification', 'STS', 'Mean (Task)', 'Mean (TaskType)', 'Bitext Mining'], inplace=True)

In [ ]:
data['Model'] = data['Model'].apply(lambda x: x.split('(')[0].strip().replace('[', '').replace(']', ''))

In [ ]:
data.isnull().sum()

In [ ]:
for col in data.columns:
    data[col] = data[col].fillna('0.00')
    if data[col].dtype == 'object':
        data[col]=data[col].astype(float)


In [ ]:
filtered = data[
    (data['Total Parameters (B)'] != 0.00) &
    (data['Total Parameters (B)'] < 0.25)# & (data['Total Parameters (B)'] < 2.0)
]
sorted_data = filtered.sort_values('Total Parameters (B)', ascending=True)


In [ ]:
sorted_data= sorted_data.sort_values('Retrieval', ascending=False)

In [ ]:
sorted_data.head(10)

In [ ]:
sorted_data.to_csv('./../filtered_sorted_models.csv', index=False)

In [ ]:
import os
os.chdir('./..')

In [ ]:
from src.utils import log, CustomException
log = log()

1. Based on research I chose bge-base-en-v1.5.
Features:
    - 0.1B parameters 
    - model size is 430MB approx.
    - 512 MAX tokens
    - Retriever effieciency is also good.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-base-en-v1.5')
tokenizer = model.tokenizer

In [ ]:
text = "What is the capital of France? The capital of France is Paris. You can also visit the Eiffel Tower in Paris."
token_count = tokenizer.encode(text)
print(f"Number of tokens: {len(token_count)}")

In [ ]:
import sys
import json
try:
    log.info("Loading cleaned EU MDR 2017-745 documents for token length analysis.")
    with open('data/processed/cleaned_eu_mdr_2017-745.json', 'r') as f:
        docs = json.load(f)
    
    structure_prefix = ""
    part_pattern = "\nPART [A-Z] \n"
    simple_pattern = r'\n(\d+)\.\s*\n'
    decimal_pattern = "\n\d+\.\d+\.\s*\n"
    triple = "\n\d+\.\d+\.\d+\.\s*\n"

    for i, doc in enumerate(docs):
        page_content = doc.get('page_content')
        metadata = doc.get('metadata')
        token_length = len(tokenizer.encode(page_content))
      

            
except Exception as e:
    log.exception(f"An error occurred: {e}")
    raise CustomException(e, sys)

In [ ]:
for doc in docs:
    doc.get('page_content', '')

In [ ]:
info = docs[148].get('page_content', '')

In [ ]:
print(info)

In [ ]:
import re
pattern = r'\n(\d+)\.\s*\n(.+)'
part_pattern = "\nPART [A-Z] \n(.+)"
matches =  re.finditer(part_pattern, docs[147].get('page_content'))

In [ ]:
len(tokenizer.encode(docs[147].get('page_content', '')))

In [ ]:
patterns = {
    "part"        : r'\nPART [A-Z] \n(.+)',
    "simple"      : r'\n(\d+)\.\s*\n(.+)',
    "decimal"     : r'(?:^|\n)(\d+\.\d+\.)(?!\d)\s+(.+)',   
    "triple"      : r'(?:^|\n)(\d+\.\d+\.\d+\.)\s+(.+)',
    "paren_letter": r'(?:^|\n)\(([a-z])\)\s+(.+)',          
    "paren_num"   : r'(?:^|\n)\((\d+)\)\s+(.+)',  
    "bullet"      : r'(?:^|\n)(—)\s+(.+)',          
}

marker_patterns = {"pattern_annex" : "^(ANNEX [IVX]+) \n(.+)", "pattern_chapter" : "^(CHAPTER [IVX]+) \n(.+)", "pattern_section" : "^(SECTION [0-9]+) \n(.+)", "pattern_article" :"^(Article [0-9]+) \n(?!Article)(?!— )(.+)"}


In [ ]:
def token_length(page_content):
    return len(tokenizer.encode(page_content))
def get_split_levels(page_content):
    levels = []
    if token_length(page_content) <= 512:
        return ['no_split']
    if re.search(patterns['part'], page_content):
        levels.append("part")
    if re.search(patterns['simple'], page_content):
        levels.append("simple")
    if re.search(patterns['decimal'], page_content):
        levels.append("decimal")
    if re.search(patterns['triple'], page_content):
        levels.append("triple")
    if re.search(patterns['paren_letter'], page_content):
        levels.append("paren_letter")
    if re.search(patterns['paren_num'], page_content):
        levels.append("paren_num")
    if re.search(patterns['bullet'], page_content):
        levels.append("bullet")
    return levels

In [ ]:
def get_text_piece(pattern, text):
    find = re.finditer(pattern, text, re.M)
    matches = []
    pieces = []
    for match in find:
        matches.append((match.start(), match.group()))
    if not matches:
        return [text]
    if matches and matches[0][0] > 0:
        pieces.insert(0, text[0:matches[0][0]])
    for i, mark in enumerate(matches):
        pattern_length = len(mark[1])
        if i < len(matches)-1:
            pieces.append(text[mark[0]:matches[i+1][0]])
        else:
            pieces.append(text[mark[0]:])
        
    return pieces

a = get_text_piece(patterns['part'], info)
lengths = [token_length(piece) for piece in a]
a


In [ ]:
any(re.match(p, x) for p in marker_patterns.values() for x in a )

In [ ]:
S = []
for s in a:
    if token_length(s) > 512:
        S.append(get_text_piece(patterns['simple'], s))
S

In [ ]:
v = any(re.match(p,g) for p in patterns.values() for k in S for g in k)
v

In [ ]:
D = []
for d in S:
    for piece in d:
        print(f"Processing piece with token length: {token_length(piece)}")
        if token_length(piece) > 512:
            D.append(get_text_piece(patterns['decimal'], piece))
D

In [ ]:
T=[]
for t in D:
    for piece in t:
        #print(f"Processing piece with token length: {token_length(piece)}")
        if token_length(piece) > 512:
            T.append(get_text_piece(patterns['triple'], piece))
T

In [ ]:
w = ["\n1.  \nDefinitions \nAutomatic identification and data capture ('AIDC') \nAIDC is a technology used to automatically capture data. AIDC technologies include bar codes, smart cards, \nbiometrics and RFID. \nBasic UDI-DI \nThe Basic UDI-DI is the primary identifier of a device model. It is the DI assigned at the level of the device unit \nof use. It is the main key for records in the UDI database and is referenced in relevant certificates and EU \ndeclarations of conformity. \nUnit of Use DI \nThe Unit of Use DI serves to associate the use of a device with a patient in instances in which a UDI is not \nlabelled on the individual device at the level of its unit of use, for example in the event of several units of the \nsame device being packaged together. \nConfigurable device \nA configurable device is a device that consists of several components which can be assembled by the \nmanufacturer in multiple configurations. Those individual components may be devices in themselves. \nConfigurable devices include computed tomography (CT) systems, ultrasound systems, anaesthesia systems, \nphysiological Monitoring systems, radiology information systems (RIS). \nConfiguration \nConfiguration is a combination of items of equipment, as specified by the manufacturer, that operate together as \na device to achieve an intended purpose. The combination of items may be modified, adjusted or customized to \nmeet specific needs. \nConfigurations include inter alia: \n—  gantries, tubes, tables, consoles and other items of equipment that can be configured/combined to deliver an \nintended function in computed tomography. \n—  ventilators, breathing circuits, vaporizers combined to deliver an intended function in anaesthesia. \nUDI-DI \nThe UDI-DI is a unique numeric or alphanumeric code specific to a model of device and that is also used as the \n'access key' to information stored in a UDI database. \nHuman Readable Interpretation ('HRI') \nHRI is a legible interpretation of the data characters encoded in the UDI carrier. \nPackaging levels \nPackaging levels means the various levels of device packaging that contain a defined quantity of devices, such as \na carton or case. \nUDI-PI \nThe UDI-PI is a numeric or alphanumeric code that identifies the unit of device production. \nThe different types of UDI-PIs include serial number, lot number, software identification and manufacturing or \nexpiry date or both types of date. \nUsefull Information to Consider:\nRadio Frequency Identification RFID \nRFID is a technology that uses communication through the use of radio waves to exchange data between \na reader and an electronic tag attached to an object, for the purpose of identification. \nShipping containers \nA shipping container is a container in relation to which traceability is controlled by a process specific to logistics \nsystems. \nUnique Device Identifier ('UDI') \nThe UDI is a series of numeric or alphanumeric characters that is created through a globally accepted device \nidentification and coding standard. It allows the unambiguous identification of a specific device on the market. \nThe UDI is comprised of the UDI-DI and the UDI-PI. \nThe word 'Unique' does not imply serialisation of individual production units. \nUDI carrier \nThe UDI carrier is the means of conveying the UDI by using AIDC and, if applicable, its HRI. \nUDI carriers include, inter alia, ID/linear bar code, 2D/Matrix bar code, RFID. "]
get_split_levels(w[0])

In [ ]:
def create_chunks(text, metadata):
    return {'page_content': text, 'metadata': metadata}

def split_document(text, metadata, levels, prefix = ""):
    if not levels or token_length(text) <= 512:
        if prefix:
            final_text = prefix + "\n" + text
        else:
            final_text = text
        new_metadata = metadata.copy()
        new_metadata['token_length'] = token_length(final_text)
 
        return [create_chunks(final_text, new_metadata)]

    current_level = levels[0]
    remaining_levels = levels[1:]
    pieces = get_text_piece(patterns[current_level],text)
    chunks = []
    #print(f"="*200 + "\nFrom Pieces: \n",repr(pieces)+ "\n", "="*200)
    main_header = ""

    for i, piece in enumerate(pieces):
        if any(re.match(pattern, piece) for pattern in marker_patterns.values()):
            continue

        match = re.match(patterns[current_level], piece)
        if match:
            header = match.group()
            new_prefix = prefix + "-" + header if prefix else "inner_Chunk_Context: " + header
            piece = piece.replace(header, "")
        else:
            new_prefix = prefix
        
        chunks.extend(split_document(piece, metadata, remaining_levels, prefix=new_prefix))
        '''total = len(chunks)
        for i, chunk in enumerate(chunks,start = 1):
            chunk['metadata']['subchunk_id'] = str(i)
            chunk['metadata']['total_subchunks'] = total'''
            
    #print(pieces)     
    return chunks

In [ ]:
info = docs[147].get('page_content', '')
meta = docs[147].get("metadata")
level = get_split_levels(info)
x = split_document(info, meta, level, prefix = "")
x

In [ ]:
pa = [p['metadata']['page_number'] for p in x]
pa

In [ ]:
all_chunks = []
for i, doc in enumerate(docs):
    page_content = doc.get('page_content', '')
    metadata = doc.get('metadata', {})
    levels = get_split_levels(page_content)
    chunks = split_document(page_content, metadata, levels)
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")
over_512 = [c for c in all_chunks if c['metadata']['token_length'] > 512]
print(f"Chunks over 512: {len(over_512)}")

In [ ]:
for c in over_512:
    print(c['metadata']['token_length'], 
          c['metadata'].get('annex',''), 
          c['metadata'].get('article',''),
          c['page_content'][:500])
    print()

In [ ]:
for chunk in all_chunks:
    if chunk['metadata']['article'] == 'Article 12':
        print(chunk['page_content'])
        print(chunk['metadata']['token_length'])

# New Approch:

Here is the summary what i have, what i want and what is current approach doing:

I already have separate section-wise chunks from pdf_extractor.

I have to take one chunk that is already splitted section or article-wise.

Process the chunk and split those chunk into 512 sub-chunks without breaking from mid-sentence.

Get the new chunk with new structural boundry follow the same process.

What current approach does:

It splits eacxh chunk by levels not by token length.

So, when level changes it creates new chunk instaed of appending it until 512 or less.

it should start at new level only if it's exceeds then 512 or 500

In [ ]:
def create_chunks(text, metadata):
    return {'page_content': text, 'metadata': metadata}

In [ ]:
def pack(chunks, max_tokens=512):
    packed = []
    buffer_text = ""
    buffer_meta=dict()
    buffer_pages = []

    def flush():
        buffer_meta['token_length'] = token_length(buffer_text)
        packed.append(create_chunks(buffer_text, buffer_meta))

    for i, c in enumerate(chunks):
        candidate = buffer_text + "\n" + c['page_content'] if buffer_text else c['page_content']
        page = c['metadata'].get('page_number')

        if token_length(candidate) > max_tokens and buffer_text:

            flush()
            buffer_text = c['page_content']
            buffer_meta = c['metadata'].copy()
            
        else:
            if not buffer_text:
                buffer_meta= c['metadata'].copy()
            buffer_text = candidate
       

    if buffer_text:
        flush()
    total = len(packed)
    for position, chunk in enumerate(packed, start=1):
        chunk['metadata']['subchunk_id'] = position
        chunk['metadata']['total_subchunks'] = total
    
    return packed

            

In [ ]:
def split_document(text, metadata, levels, prefix=""):
    if token_length(text) <= 512:
        if prefix:
            final_text = prefix + "\n" + text
        else:
            final_text = text
        
        new_metadata = metadata.copy()
        new_metadata['token_length'] = token_length(final_text)
        return [create_chunks(final_text, new_metadata)]
    
    if not levels:
        if prefix:
            final_text = prefix + "\n" + text
        else:
            final_text = text
        new_metadata = metadata.copy()
        new_metadata['token_length'] = token_length(final_text)
        return [create_chunks(final_text, new_metadata)]
    
    current_level = levels[0]
    remaining_levels = levels[1:]

    text_pieces = get_text_piece(patterns[current_level], text)
    if not text_pieces:
        text_pieces = [text]
    sub_chunks = []
    for i, piece in enumerate(text_pieces):
        '''if any(re.match(pattern, piece) for pattern in marker_patterns.values()):
            continue'''

        match = re.match(patterns[current_level],piece)
        
        if match:
            header = match.group()
            new_prefix = prefix + "-" + header if prefix else header
            piece = piece.replace(header, "")
        
        else:
            new_prefix = prefix

        sub_chunks.extend(split_document(piece, metadata, remaining_levels, prefix=new_prefix))
    return pack(sub_chunks, max_tokens=512)

In [ ]:
all_chunks = []
for i, doc in enumerate(docs):
    page_content = doc.get('page_content', '')
    metadata = doc.get('metadata', {})
    levels = get_split_levels(page_content)
    chunks = split_document(page_content, metadata, levels)
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")
over_512 = [c for c in all_chunks if c['metadata']['token_length'] > 512]
print(f"Chunks over 512: {len(over_512)}")

In [ ]:
all_chunks[1]

In [ ]:
for c in over_512:
    print(c['metadata']['token_length'], 
          c['metadata'].get('annex',''), 
          c['metadata'].get('article',''),
          c['page_content'][:500])
    print()

In [ ]:
def is_header_only_doc(doc):
    body = doc['page_content'].strip()
    m = doc['metadata']

    def field(key):
        v = m.get(key, '')
        return str(v) if v is not None else ''

    title_only = body in (
        (field('section') + field('section_title')).strip(),
        (field('chapter') + field('chapter_title')).strip(),
        (field('annex')   + field('annex_title')).strip(),
    )
    bare = re.match(
        r'^(SECTION \d+|CHAPTER [IVX]+|ANNEX [IVX]+)\s*\n[A-Za-z][\w\s,&-]*\s*$',
        body
    )
    return bool(title_only or (token_length(body) < 20 and bare))

docs = [d for d in all_chunks if not is_header_only_doc(d)]

In [ ]:
p = []

for i, c in enumerate(docs):
    d = {
        "Chunk_no": i,
        "token_length": c["metadata"]["token_length"]
    }
    p.append(d)

In [ ]:
data = pd.DataFrame(p)
import numpy as np

bins = np.arange(0, 550, 20)

data['token_length'].plot(kind='hist', bins=bins, edgecolor='black',xlabel="token_length")



In [ ]:
with open('data/processed/chunks/chunks.json', 'w') as f:
    json.dump(docs, f, indent=4)

In [ ]:
import json

with open('data/processed/chunks/chunks.json', 'r') as f:
    docs = json.load(f)

In [ ]:
print(docs)

# Build the Embedding setup

In [ ]:
import chromadb

## Creating the persistent database

In [ ]:
client  = chromadb.PersistentClient(path = "./data/database/chromadb/")

In [ ]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import torch
try:
    device = "cuda" if torch.cuda.is_available() else 'cpu'
    embedding_fn = SentenceTransformerEmbeddingFunction(model_name='BAAI/bge-base-en-v1.5', device=device)
except Exception as e:
    log.exception(f"An Error occured: {e}")
    raise CustomException(e, sys)

In [ ]:
#client.delete_collection(name="mdr_1")
collection = client.get_or_create_collection(name = "mdr_1", 
                                     embedding_function=embedding_fn,
                                     configuration={
                                                "hnsw": {
                                                    "space": "cosine",
                                                    "ef_construction": 300
                                                }
                                            })

In [ ]:
collection.configuration

### Evaluating Small

* Add few chunks to collection
* Evaluate with a query and check the semantic match
* check what it returns: Distance or similarity?

In [ ]:
import hashlib
ids, pcs, metadatas = [],[],[]
for doc in docs:
    metadata = doc.get('metadata')
    metadatas.append(metadata)
    pc = doc.get('page_content')
    pcs.append(pc)
    chunk_id = hashlib.sha256(pc.encode('utf-8')).hexdigest()
    ids.append(chunk_id)

collection.add(ids=ids, 
                   documents=pcs,
                   metadatas=metadatas)


In [ ]:
import hashlib
ids1, pcs, metadatas = [],[],[]
for doc in docs:
    metadata = doc.get('metadata')
    metadatas.append(metadata)
    pc = doc.get('page_content')
    pcs.append(pc)
    chunk_id = hashlib.sha256(pc.encode('utf-8')).hexdigest()
    ids1.append(chunk_id)


    

In [ ]:
print(len(ids), len(ids1))
print("identical:", ids == ids1)

In [ ]:
len(ids) == len(set(ids))

### Confirm the colletion count

In [ ]:
collection.count()

### Confirm the embedding dimensions

In [ ]:
# Pullng one embedding
collection.get(include=['embeddings'], ids=["2fa14721893766c74fe238401df190da4d802fd600f9fccdcda37781ce6143e4"]).get('embeddings').shape

### Querying from the added chunks to get the self-match distance

In [ ]:
query_1 = collection.query(
    query_texts=["Why did the EU replace the old medical device directives?"],
    n_results=3
)

In [ ]:
query_1.get('metadatas')

In [ ]:
query_1.get('distances')

### Querying with negative query(unrelated question)

In [ ]:
query_2 = collection.query(
    query_texts=["What are the UDI carrier placement rules for reusable devices?"],
    n_results=3
)

In [ ]:
query_2.get('metadatas')

In [ ]:
query_2.get('distances')

## Observations:
1. The collection is persistent and store the chunks efficiently.
2. The Embedding function works well and generates expected embeddings dimensions which is 768.
3. The chromadb's retrieval fetches the semantically matched information and gices efficient distance scores.


## Scalling to all the chunks
* Delete and rebuild the collections

* Build the lists across all the chunks
 
* Assert Uniqueness

* Add all chunks to collection

* Confirm the collection count

### Delete-and-rebuild collection

In [8]:
import chromadb
client  = chromadb.PersistentClient(path = "./data/database/chromadb/")

In [6]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
import torch
try:
    device = "cuda" if torch.cuda.is_available() else 'cpu'
    embedding_fn = SentenceTransformerEmbeddingFunction(model_name='BAAI/bge-base-en-v1.5', device=device)
except Exception as e:
    log.exception(f"An Error occured: {e}")
    raise CustomException(e, sys)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
#client.delete_collection(name="mdr_1")
collection = client.get_or_create_collection(name = "eu_mdr", 
                                     embedding_function=embedding_fn,
                                     configuration={
                                                "hnsw": {
                                                    "space": "cosine",
                                                    "ef_construction": 300
                                                }
                                            })

### Build the lists across all the chunks

In [10]:
len(docs)

369

In [11]:
import hashlib

try:
    ids, metadatas, pcs= [], [], []
    for doc in docs:
        pc = doc.get('page_content')
        pcs.append(pc)
        metadatas.append(doc.get('metadata'))
        ids.append(hashlib.sha256(pc.encode("utf-8")).hexdigest())

except Exception as e:
    log.exception(f"The Error occured: {e, sys}")
    raise CustomException(e,sys)

In [12]:
# Confirm the total docs
len(ids)

369

### Assert Uniqueness

In [13]:
len(ids) == len(set(ids))

True

### Add to collection

In [14]:
if len(ids) == len(set(ids)):
    collection.add(ids=ids, 
                   documents=pcs,
                   metadatas=metadatas)

### Assert the count

In [ ]:
collection.count()

### Questions:

1. Under what conditions are medical devices manufactured and used within health institutions exempt from the requirements of the regulation? Gold chunk that contains article 5
2. Compare the product categories in Annex XVI that involve modifications to the human body. Which categories involve surgical invasion, injections, electromagnetic radiation, and brain stimulation, and what are their respective purposes?
3. What shall be included by the PMCF plan?
4. What are the requirements mentioned in Article 62 for conformity of devices?
5. How can a manufacturer obtain proof that a CE-marked device can be marketed in the European Union for export purposes?(similar to previous question)
6. what is 'notified body'?
7. What are the differences in conformity assessment between Class IIa and Class IIb devices?
8. What is the definition of a medical device under MDR?
9. Where does the 'Regulation (EU) 2017/745' not apply?
10. which documents are required if one wants to perform clinincal invastigation?

In [ ]:
query_3 = collection.query(
    query_texts=["which documents are required if one wants to perform clinincal invastigation?"],
    n_results=5
)

In [ ]:
query_3

In [ ]:
eval_questions = ["Under what conditions are medical devices manufactured and used within health institutions exempt from the requirements of the regulation? Gold chunk that contains article 5",
"Compare the product categories in Annex XVI that involve modifications to the human body. Which categories involve surgical invasion, injections, electromagnetic radiation, and brain stimulation, and what are their respective purposes?",
"What shall be included by the PMCF plan?",
"What are the requirements mentioned in Article 62 for conformity of devices?",
"How can a manufacturer obtain proof that a CE-marked device can be marketed in the European Union for export purposes?",
"what is 'notified body'?",
"What are the differences in conformity assessment between Class IIa and Class IIb devices?",
"What is the definition of a medical device under MDR?",
"Where does the 'Regulation (EU) 2017/745' not apply?",
"which documents are required if one wants to perform clinincal invastigation?"]

container = []

for question in eval_questions:
    
    query = collection.query(
    query_texts=[question],
    n_results=5
    )
    
    container.append({"question": question,
        "retrieved_data": query}
                )

with open("data/evaluation/test_questions.json", 'w') as f:
    json.dump(container, f, indent=4)

In [ ]:
container

## Building Evaluation

In [ ]:
def recall_at_k(retrieved_ids, gold_chunk_ids, k):
    rid_set = set(retrieved_ids[:k])
    gid_set = set(gold_chunk_ids)
    if not len(gid_set & rid_set) == 0:
        return 1
    else: return 0

In [ ]:
def RR(retrieved_ids, gold_chunk_ids):
    rank=None
    for i, rid in enumerate(retrieved_ids):
        if rid in gold_chunk_ids:
            rank = i+1
            break
    if rank is not None:    
        return 1 / rank

    return 0

In [ ]:
from pathlib import Path
evalution_set = Path("data/evaluation/test_questions.json")


with open(evalution_set, 'r') as f:
    eval_data = json.load(f)

hits = 0
reciprocal_ranks = []
for questions in eval_data['questions']:
    question = questions['question']
    results = collection.query(query_texts=[question], n_results=5)
    gold_chunk_id = questions['gold_chunk_ids']
    retrieved_ids = results['ids'][0]
    recall = recall_at_k(retrieved_ids, gold_chunk_id, k=1)
    if recall:
        hits += 1
    reciprocal_ranks.append(RR(retrieved_ids, gold_chunk_id))

   

recall_at_1  = hits/len(eval_data['questions'])
MRR = sum(reciprocal_ranks)/len(eval_data['questions'])


In [ ]:
recall_at_1, MRR

In [ ]:
def label(number, title):
    if title:
        return f"{number} ({title})"
    return number

In [ ]:
def create_source(metadata: dict):
    article= metadata.get('article', "").strip()
    annex = metadata.get("annex","").strip()
    chapter = str(metadata.get('chapter',"")).strip()
    chapter_title = metadata.get("chapter_title", "").strip()
    annex_title = metadata.get("annex_title","").strip()
    article_title = metadata.get("article_title", "").strip()
    section = metadata.get("section","").strip()
    section_title = metadata.get("section_title").strip()

    if chapter == "0" and chapter_title == "preamble":
        source = "EU MDR Preamble"
    
    elif article:
        parts = []
        if chapter: parts.append(label(chapter, chapter_title))
        if section: parts.append(label(section,section_title))
        parts.append(label(article, article_title))
        source =  ", ".join(parts)

    elif annex:
        parts = [label(annex, annex_title)]
        if chapter: parts.append(label(chapter, chapter_title))
        source =  ", ".join(parts)

    return source



In [ ]:
for x in container:
    f = x.get("retrieved_data")
    o = f['metadatas'][0][0]
    print(create_source(o))

In [ ]:
query_3

In [ ]:
query_3['documents'][0]

In [ ]:
# Building a function

def create_model_context(retrieved_chunks: dict) -> str:
    list_of_chunks = []
    docs = retrieved_chunks['documents'][0]
    metadatas = retrieved_chunks['metadatas'][0]
    for content, meta in zip(docs, metadatas):
        source = create_source(meta)
        page_c = "<CHUNK_SOURCE: " + source + ">\n" + content + "</CHUNK>"
        list_of_chunks.append(page_c)
    
    context = "\n-------------------------------------------------------------------------------------------------\n".join(list_of_chunks)
    return context


create_model_context(query_3)

In [ ]:
test_generator = []
for q in container:
    question = q['question']
    result = collection.query(
        query_texts=[question]
    )
    test_generator.append(
        {'question': question,
         'context': create_model_context(result)}
    )


test_generator

In [ ]:
with open("prompt.md", 'r') as f:
    prompt = f.readlines()

prompt.insert(10, 'hi')
prompt

### Building a function to merge prompt and retrieved chunks

In [ ]:
def create_prompt(md_file, question):
    results = collection.query(
        query_texts=[question]
    )
    model_context = create_model_context(results)
    with open(md_file, 'r') as f:
        prompt = f.readlines()
    index = prompt.index("<SOURCES>\n")
    prompt.insert(index+1, model_context)
    prompt.insert(len(prompt), f"<QUESTION:>\n{question} \n</QUESTION>")

    with open(md_file,'w') as c:
        c.writelines("".join(prompt))


In [ ]:
create_prompt("prompt.md", "what is Medical Device?")